# Extracting ECG data from DICOM files
The ecgprocess.py module wraps pydicom to standardise and extract patient information tables, as well as lead-specific waveforms and median beats.

In [1]:
import os
import tempfile
import numpy as np
import pandas as pd
import ecgprocess.process_dicoms as pdicom
from ecgprocess.process_dicoms import (
    ECGDICOMReader,
    ECGDICOMTable,

)
from ecgprocess.example_data.examples import (
    list_data_paths,
)

# Example file
dicom_file_path = list_data_paths()['example_dicom_2']


## The ECGDICOMReader dataclass 

In [2]:
ECG_DCM = ECGDICOMReader(retain_raw=False)
dicom_record = ECG_DCM(path=dicom_file_path)

In [3]:
dir(dicom_record)

['ACQUISITION_DATE_SOURCE',
 'ACQUISITION_DATE_TARGET',
 'ECG_TRAIT_DICT',
 'GeneralInfo',
 'MEDIAN_BEATS_DICT',
 'METADATA',
 'MedianWaveformPresent',
 'MedianWaveforms',
 'SOPinstanceUID',
 'STUDY_DATE_SOURCE',
 'STUDY_DATE_TARGET',
 'WAVE_FORMS',
 'WAVE_FORMS_DICT',
 'Waveforms',
 '__annotations__',
 '__call__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_get_lead_info',
 '_resampling_500hz',
 'augment_leads',
 'make_leadvoltages',
 'oversampled',
 'resample_500',
 'retain_raw']

### Participant information

In [4]:
# patient information
pd.Series(dicom_record.GeneralInfo)

SOPinstanceUID               1.3.6.1.4.1.40744.65.2218354498692802781165553...
SERIESinstanceUID            1.3.6.1.4.1.40744.65.2534925851648250593315477...
STUDYinstanceUID             1.3.6.1.4.1.40744.65.1513822871218815858359656...
PatientID                                                                66954
StudyDate                                                           2006-02-01
StudyTime                                                               102403
AccessionNumber                                                     ST20060201
PatientBirthDate                                                              
PatientName                                                    (6, 6, 9, 5, 4)
PatientSex                                                                    
StudyDescription                                                          DT4H
AcquisitionDateTime                                        2006-02-01 10:24:03
AcquisitionTimeZone                                 

### Waveforms and Median beats

In [5]:
dicom_record.Waveforms

{'I (Einthoven)': array([-105., -105., -105., ...,  -75.,  -80.,  -85.]),
 'II': array([-185., -185., -185., ..., -215., -220., -220.]),
 'III': array([ -75.,  -75.,  -75., ..., -140., -140., -135.]),
 'aVR': array([145., 145., 145., ..., 145., 150., 155.]),
 'aVL': array([-10., -10., -10., ...,  25.,  25.,  20.]),
 'aVF': array([-130., -130., -130., ..., -180., -185., -180.]),
 'V1': array([65., 65., 65., ..., 40., 45., 45.]),
 'V2': array([-175., -175., -175., ..., -250., -245., -240.]),
 'V3': array([ -55.,  -55.,  -55., ..., -155., -150., -145.]),
 'V4': array([-210., -210., -210., ..., -125., -125., -125.]),
 'V5': array([-145., -145., -145., ..., -165., -165., -165.]),
 'V6': array([-125., -125., -125., ..., -190., -185., -185.])}

In [6]:
dicom_record.MedianWaveforms['II'][0:20]

array([20., 15., 10., 10., 10., 10.,  5.,  0.,  0.,  0.,  0.,  5.,  5.,
        5.,  5.,  5.,  0.,  0.,  0.,  0.])

## Custom behaviour 
The class object has a number of attributes (dictionaries) which can be used to extract more (or less) data from pydicom instance. 

In [7]:
# extract `any comments` - which is rather uninformative in this case: `nan`.  

# add the comments name to the METADATA dict
# here the value should map to the pydicom attribute name 
# and the key to the new name we want to use internally
ECG_DCM.METADATA['Image Comment'] = 'ImageComments'
dicom_record_new = ECG_DCM(path=dicom_file_path)
dicom_record_new.GeneralInfo['Image Comment'] 

nan

## ECGDICOMTable 
A class to extract data from multiple records and map these to tabular data.

We will first simply initilise the class and differentiate between valid and invalid files. 

In [8]:

dicom_path_list = [
    list_data_paths()['example_dicom_1'],
    list_data_paths()['example_dicom_2'],
]
ecgreader = ECGDICOMReader()
ecgtable = ECGDICOMTable(ecgreader, path_list=dicom_path_list)
res = ecgtable(verbose=True)
print('Failed paths: {0}\nCurated paths: {1}.'.format(
    None if len(res.FailedPathList) == 0 else res.FailedPathList, 
    res.CuratedPathList))

Failed paths: None
Curated paths: ['/home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm', '/home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm'].


### Process the two files
Our first try at this will raise an error because the headers do not match. 

In [9]:
try:
    res.get_table()
except Exception as e:
    print(f"\n\nAn error occurred: {e}")

Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm.
Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm.


An error occurred: "The dictionaries contain distinct keys. The last set of valid keys was ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6'], compared to ['I (Einthoven)', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']."


Next we will solve this by providing a dictionary mapping the old name to the new name. These dictionaries can be expanded for multiple potential names and ignored if there are no matches. This will results in a long table with multiple files row-binded.

In [10]:
combined=res.get_table(update_keys={'I (Einthoven)': 'I'})
combined.WaveFormsTable

Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm.
Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm.


,SOPinstanceUID,SamplingSequence,Lead,Voltage,Waveform type
0,2.25.269796857626990821315969488216511468638,0,I,0.00,rhythm
1,2.25.269796857626990821315969488216511468638,1,I,0.00,rhythm
2,2.25.269796857626990821315969488216511468638,2,I,29.28,rhythm
3,2.25.269796857626990821315969488216511468638,3,I,43.92,rhythm
4,2.25.269796857626990821315969488216511468638,4,I,48.80,rhythm
...,...,...,...,...,...
119995,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4995,V6,-195.00,rhythm
119996,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4996,V6,-190.00,rhythm
119997,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4997,V6,-190.00,rhythm
119998,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4998,V6,-185.00,rhythm


### Write to disk
These tables can of course be written to disk. However, loading data from multiple files to simply write these to disk will be ineffecient. A more memory efficient solution is to extract each individual DICOM and append results one-at-a-time to a directory file. This is what `write_ecg` does ... Actually it writes four files to disk!

In [11]:
with tempfile.TemporaryDirectory() as temp_dir:
    combined=res.write_ecg(update_keys={'I (Einthoven)': 'I'}, target_path=temp_dir)
    # list saved files
    print('Directory content:')
    for file_name in os.listdir(getattr(combined, 'target_path')):
        print('   - {}'.format(file_name))

Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm.
Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm.
Directory content:
   - GeneralInfoTable.tsv.gz
   - WaveFormsTable.tsv.gz
   - FailedFiles.txt
   - MedianWaveTable.tsv.gz


In [12]:
help(pdicom)

Help on module ecgprocess.process_dicoms in ecgprocess:

NAME
    ecgprocess.process_dicoms

DESCRIPTION
    A module to extract median beats and raw waveforms from DICOM ECGs wrapping
    pydicom.

CLASSES
    builtins.object
        BaseECGDICOMReader
            ECGDICOMReader
        ECGDICOMTable

    class BaseECGDICOMReader(builtins.object)
     |  BaseECGDICOMReader(lead_voltages=None, lead_voltages2=None, results_dict=None)
     |
     |  An ECGDICOMReader base class implementing the more efficient __slots__ for
     |  the waveform arrays, while still retaining __dict__ dynamic attribute
     |  creation.
     |
     |  Methods defined here:
     |
     |  __init__(self, lead_voltages=None, lead_voltages2=None, results_dict=None)
     |      Initialises slots entries to `None`.
     |
     |  ----------------------------------------------------------------------
     |  Data descriptors defined here:
     |
     |  GeneralInfo
     |
     |  MedianWaveforms
     |
     |  Wav